# 9-element UniversalXAS / Tuned-UniversalXAS sweep (all-in-one notebook)

Runs where the ZN notebook already exported 64-D features for all 9 elements
(Ti, V, Cr, Mn, Fe, Co, Ni, Cu, Zn).

Design:
- Stage 1: 30 UniversalXAS sweeps; pick best by median validation MSE.
- Stage 2: 100 Tuned sweeps per element (~900 total) from the best Universal checkpoint.
- Stage 3: evaluate best Universal + best Tuned on val and test (test once).

No edit needed; path hardcoded from the ZN notebook (run zn_full).


In [ ]:
import json as _json, math, random
from copy import deepcopy
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from omnixas.model.xasblock import XASBlock

ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn']
TASKS = [e + '_FEFF' for e in ELEMENTS]
DIM = 64; SPEC = 141

# 9-element features exported by the ZN notebook (train_zn_expert_and_full.ipynb): output/training/zn_expert_study/zn_full/features
FEATURES_DIR = Path(r'output/training/zn_expert_study/zn_full/features')
OUTPUT = Path(r'output/training/zn_sweep') / 'zn_universal_tuned'
OUTPUT.mkdir(parents=True, exist_ok=True)

def load(task, split):
    x = np.atleast_2d(np.loadtxt(FEATURES_DIR / (task + '_' + split + '_X.txt'), dtype=np.float32))
    y = np.atleast_2d(np.loadtxt(FEATURES_DIR / (task + '_' + split + '_y.txt'), dtype=np.float32))
    return x, y

train_base = np.array([
    float(np.median(((load(t, 'train')[1] - load(t, 'train')[1].mean(0)) ** 2).mean(1)))
    for t in TASKS
])
print('train bases:', dict(zip(ELEMENTS, [round(float(b), 6) for b in train_base])))


In [ ]:
def train_head(X, y, vX, vY, source, args, *, lr, epochs, schedule, es_metric, label,
                 hidden_dims, dropout=0.05, loss_balance_power=0.0,
                 train_task=None, val_task=None, task_base=None, loss_scale=None, seed=None):
    seed = args.seed if seed is None else seed
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with patch.object(XASBlock, 'DROPOUT', dropout):
        head = XASBlock(DIM, list(hidden_dims), SPEC).to(dev)
    if source is not None: head.load_state_dict(source, strict=True)
    Xt, yt = torch.as_tensor(X, device=dev), torch.as_tensor(y, device=dev)
    vXt, vYt = torch.as_tensor(vX, device=dev), torch.as_tensor(vY, device=dev)
    tt = None if train_task is None else torch.as_tensor(train_task, dtype=torch.long, device=dev)
    vt = None if val_task is None else torch.as_tensor(val_task, dtype=torch.long, device=dev)
    tb = None if task_base is None else torch.as_tensor(task_base, dtype=torch.float32, device=dev)
    opt = torch.optim.Adam(head.parameters(), lr=lr, fused=dev.type == 'cuda')
    if schedule == 'plateau':
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=16, min_lr=1e-6)
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=500, eta_min=1e-6)
    rollback = source is not None
    best = float('inf'); stale = 0; best_state = deepcopy(head.state_dict())
    acc_w = deepcopy(head.state_dict()); acc_o = deepcopy(opt.state_dict())
    if rollback:
        with torch.inference_mode():
            err = (head(vXt) - vYt).square().mean(1)
        best = float(err.median())
        best_state = deepcopy(head.state_dict()); acc_w = deepcopy(head.state_dict()); acc_o = deepcopy(opt.state_dict())
    for _ in tqdm(range(epochs), desc=label, unit='epoch', leave=False):
        head.train()
        idx = torch.randperm(len(Xt), device=dev)
        for b in idx.split(args.batch_size):
            opt.zero_grad(set_to_none=True)
            pr = head(Xt[b]); rm = (pr - yt[b]).square().mean(1)
            if loss_balance_power:
                loss = torch.stack([rm[tt[b] == t].mean() / tb[t].clamp_min(1e-12).pow(loss_balance_power) for t in torch.unique(tt[b])]).mean()
            else:
                loss = rm.mean()
            if loss_scale: loss = loss / loss_scale
            loss.backward(); opt.step()
        head.eval()
        with torch.inference_mode():
            err = (head(vXt) - vYt).square().mean(1)
        metric = float(err.median())
        if metric < best:
            best, stale, best_state = metric, 0, deepcopy(head.state_dict())
            if rollback:
                acc_w = deepcopy(head.state_dict()); acc_o = deepcopy(opt.state_dict())
        else:
            stale += 1
            if rollback:
                head.load_state_dict(acc_w, strict=True)
                rates = [g['lr'] for g in opt.param_groups]
                opt.load_state_dict(deepcopy(acc_o))
                for g, r in zip(opt.param_groups, rates): g['lr'] = r
        sched.step(metric) if schedule == 'plateau' else sched.step()
        if stale >= args.head_patience: break
    return best_state, best, {'best_val_mse': best, 'hidden_dims': list(hidden_dims), 'dropout': dropout}


In [ ]:
def balanced(train, val, seed):
    n = max(len(X) for X, _ in train.values()); rng = np.random.default_rng(seed)
    xs, ys, ts = [], [], []
    for i, t in enumerate(TASKS):
        X, y = train[t]; ii = rng.integers(len(X), size=n)
        xs.append(X[ii]); ys.append(y[ii]); ts.append(np.full(n, i))
    vx = np.concatenate([val[t][0] for t in TASKS]); vy = np.concatenate([val[t][1] for t in TASKS])
    vt = np.concatenate([np.full(len(val[t][1]), i) for i, t in enumerate(TASKS)])
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(ts), vx, vy, vt

train = {t: load(t, 'train') for t in TASKS}
val = {t: load(t, 'val') for t in TASKS}
test = {t: load(t, 'test') for t in TASKS}

HIDDEN = [(500, 500, 550), (600, 600, 400), (400, 400, 400), (300, 300, 300)]
DROPOUTS = [0.05, 0.10]; LBPS = [0.0, 1.5]
args = SimpleNamespace(seed=42, batch_size=4096, head_patience=60, universal_epochs=800, tuned_epochs=1000)
rng = np.random.default_rng(args.seed + 1000)

ux, uy, utt, uvx, uvy, uvt = balanced(train, val, args.seed)
ubest = None; ures = []
for r in range(1, 31):
    h = {
        'lr': float(np.exp(rng.uniform(math.log(1e-4), math.log(5e-3)))),
        'hidden_dims': list(HIDDEN[int(rng.integers(len(HIDDEN)))]),
        'dropout': float(DROPOUTS[int(rng.integers(len(DROPOUTS)))]),
        'loss_balance_power': float(LBPS[int(rng.integers(len(LBPS)))]),
        'seed': int(rng.integers(1_000_000)),
    }
    st, m, meta = train_head(ux, uy, uvx, uvy, None, args,
                             lr=h['lr'], epochs=args.universal_epochs, schedule='plateau',
                             es_metric='median', label='Universal ' + str(r) + '/30',
                             hidden_dims=h['hidden_dims'], dropout=h['dropout'],
                             loss_balance_power=h['loss_balance_power'],
                             train_task=utt, val_task=uvt, task_base=train_base, seed=h['seed'])
    ures.append({'run': r, **h, 'val_median_mse': float(m), **meta})
    if ubest is None or m < ubest[0]: ubest = (m, st, meta, h)
    lr_v = h['lr']; hd_v = h['hidden_dims']
    print('universal ' + str(r) + '/30 val_median_mse=' + f'{m:.4e}' + ' lr=' + f'{lr_v:.2e}' + ' hidden=' + str(hd_v), flush=True)

um, ust, umeta, uhyp = ubest
torch.save(ust, OUTPUT / 'best_universal.pt')
pd.DataFrame(ures).to_csv(OUTPUT / 'universal_records.csv', index=False)
lr_u = uhyp['lr']; hd_u = uhyp['hidden_dims']; do_u = uhyp['dropout']; lbp_u = uhyp['loss_balance_power']; sd_u = uhyp['seed']
print('BEST UNIVERSAL val_median_mse=' + f'{um:.4e}' + ' lr=' + f'{lr_u:.2e}' + ' hidden=' + str(hd_u) + ' dropout=' + str(do_u) + ' lbp=' + str(lbp_u) + ' seed=' + str(sd_u), flush=True)


In [ ]:
rng = np.random.default_rng(args.seed + 2000)
tbest = None; tres = []
hdim, hdrop = umeta['hidden_dims'], umeta['dropout']
for t in TASKS:
    trX, trY = train[t]; vaX, vaY = val[t]
    for r in range(1, 101):
        hp = {
            'lr': float(np.exp(rng.uniform(math.log(1e-4), math.log(2e-3)))),
            'schedule': ['plateau', 'cosine'][int(rng.integers(2))],
            'loss_normalize': bool(rng.integers(2)),
            'seed': int(rng.integers(1_000_000)),
        }
        es = 'median_relative' if hp['loss_normalize'] else 'median'
        ls = float(train_base[TASKS.index(t)]) if hp['loss_normalize'] else None
        st, m, meta = train_head(trX, trY, vaX, vaY, ust, args,
                                 lr=hp['lr'], epochs=args.tuned_epochs, schedule=hp['schedule'],
                                 es_metric=es, label='Tuned ' + t + ' ' + str(r) + '/100',
                                 hidden_dims=hdim, dropout=hdrop, loss_scale=ls, seed=hp['seed'])
        tres.append({'element': t, 'run': r, **hp, 'val_median_mse': float(m), **meta})
        if tbest is None or m < tbest[0]: tbest = (m, st, meta, hp, t)
        lr_t = hp['lr']; sc_t = hp['schedule']; no_t = hp['loss_normalize']
        print('tuned ' + t + ' ' + str(r) + '/100 val_median_mse=' + f'{m:.4e}' + ' lr=' + f'{lr_t:.2e}' + ' sched=' + str(sc_t) + ' norm=' + str(no_t), flush=True)

tm, tst, tmeta, thyp, ttask = tbest
torch.save(tst, OUTPUT / 'best_tuned.pt')
pd.DataFrame(tres).to_csv(OUTPUT / 'tuned_records.csv', index=False)
lr_tn = thyp['lr']; sc_tn = thyp['schedule']; no_tn = thyp['loss_normalize']; sd_tn = thyp['seed']
print('BEST TUNED element=' + ttask + ' val_median_mse=' + f'{tm:.4e}' + ' lr=' + f'{lr_tn:.2e}' + ' sched=' + str(sc_tn) + ' norm=' + str(no_tn) + ' seed=' + str(sd_tn), flush=True)


In [ ]:
def evaluate(head, task):
    trX, trY = train[task]; vaX, vaY = val[task]; teX, teY = test[task]
    out = {}
    for split, (vx, vy) in (('val', (vaX, vaY)), ('test', (teX, teY))):
        head.eval()
        with torch.inference_mode():
            pr = torch.cat([head(b) for b in torch.from_numpy(vx).split(args.batch_size)]).cpu().numpy()
        err = ((pr - vy) ** 2).mean(1)
        base = float(np.median(((vy - trY.mean(0)) ** 2).mean(1)))
        med = float(np.median(err))
        out[split] = {'mse': float(err.mean()), 'median_mse': med,
                      'baseline_median_mse': base, 'eta': base / med if med > 0 else float('inf')}
    return out

with patch.object(XASBlock, 'DROPOUT', hdrop): uh = XASBlock(DIM, hdim, SPEC)
uh.load_state_dict(ust, strict=True)
with patch.object(XASBlock, 'DROPOUT', hdrop): th = XASBlock(DIM, hdim, SPEC)
th.load_state_dict(tst, strict=True)

rows = []
for t in TASKS:
    for split, m in evaluate(uh, t).items():
        rows.append({'variant': 'UniversalXAS', 'element': t, 'split': split, **m})
    for split, m in evaluate(th, t).items():
        rows.append({'variant': 'Tuned-UniversalXAS', 'element': t, 'split': split, **m})

res = pd.DataFrame(rows)
res.to_csv(OUTPUT / 'final_evaluation.csv', index=False)

# Concise table: all 9 elements, UniversalXAS vs Tuned-UniversalXAS, val and test.
tbl = []
for t in ELEMENTS:
    task = t + '_FEFF'
    row = {'element': t}
    for variant in ('UniversalXAS', 'Tuned-UniversalXAS'):
        for split in ('val', 'test'):
            m = res[(res.variant == variant) & (res.element == task) & (res.split == split)].iloc[0]
            row[variant + ' ' + split + ' median_mse'] = float(m.median_mse)
            row[variant + ' ' + split + ' eta'] = float(m.eta)
    tbl.append(row)
table = pd.DataFrame(tbl)
display(table.round(4))

summary = {
    'best_universal': {'val_median_mse': float(um), **uhyp},
    'best_tuned': {'element': ttask, 'val_median_mse': float(tm), **thyp},
}
(OUTPUT / 'summary.json').write_text(_json.dumps(summary, indent=2))
print(_json.dumps(summary, indent=2))
print('artifacts:', sorted(p.name for p in OUTPUT.iterdir()))
